# Unsupervised Regime Detection via Hidden Markov Models

## Market Microstructure Motivation

Volatility is entirely unobservable. While we can compute realized variance over a rolling window, this creates a lagging indicator. Markets typically transition between distinct macroeconomic or liquidity "regimes"—periods of relative calm and periods of violent structural breaks.

To build an optimal position-sizing signal, we need an unsupervised method to detect the *probability* of being in a volatile regime in real-time. A Hidden Markov Model (HMM) allows us to probabilistically classify these regimes purely from the log-returns without manual thresholds.


## Mathematical Derivation

Let $z_t \in \{0, 1\}$ be the hidden regime (0: Quiet, 1: Volatile). Let $y_t$ be the observed log-return.
Our emissions are Gaussian: $P(y_t | z_t = i) = \mathcal{N}(y_t; \mu_i, \sigma_i^2)$.

**1. Online Filtering (Forward Algorithm)**
To compute the real-time probability of the regime $P(z_t | y_{1:t})$, we recursively apply:
$$\hat{\alpha}_t(i) = \sum_j A(j, i) \alpha_{t-1}(j)$$
$$\alpha_t(i) \propto \hat{\alpha}_t(i) \cdot P(y_t | z_t=i)$$

**2. Offline Parameter Calibration (Baum-Welch EM)**
To fit the transition matrix $A$ and the emission parameters $(\mu_i, \sigma_i)$, we use the offline E-M loop.

*E-Step (Forward-Backward)*:
Computes the smoothed posterior $\gamma_t(i) = P(z_t=i | y_{1:T})$ and the joint state probability $\xi_t(i,j) = P(z_t=i, z_{t+1}=j | y_{1:T})$.

*M-Step*:
We maximize the expected log-likelihood by updating parameters:
$$A(i, j) = \frac{\sum_{t=1}^{T-1} \xi_t(i, j)}{\sum_{t=1}^{T-1} \gamma_t(i)}$$
$$\sigma_i^2 = \frac{\sum_{t=1}^T \gamma_t(i) (y_t - \mu_i)^2}{\sum_{t=1}^T \gamma_t(i)}$$


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from benchmark_hmm import run_benchmark

# Train the HMM via Baum-Welch and plot the online posterior
run_benchmark()
